# Semaine 3 — Jour 5 — Sharing State

Notebook étudiant généré depuis le Markdown source.

## Objectifs

- Distinguer état local, shared state, conversation state et memory.
- Implémenter un store partagé versionné.
- Filtrer les snapshots par permission.
- Créer un handoff minimal.

## Contexte

Dans un workflow multi-agent, le shared state est le contrat qui permet aux agents de collaborer sans dépendre d'une mémoire globale implicite.

In [ ]:
from pathlib import Path
import sys

lab_path = Path.cwd()
candidate_paths = [
    lab_path / "book" / "week03" / "day05" / "labs",
    lab_path.parent / "book" / "week03" / "day05" / "labs",
    lab_path.parent.parent / "book" / "week03" / "day05" / "labs",
]
for candidate in candidate_paths:
    if candidate.exists():
        sys.path.insert(0, str(candidate))
        break

from shared_state_store import SharedStateStore, ConflictError, PermissionDenied, run_demo

## Démonstration

In [ ]:
demo = run_demo()
demo["summary"]

## Exercice guidé

Créez une clé privée, une clé partagée, puis un handoff filtré.

In [ ]:
store = SharedStateStore()
store.write(agent="triage", key="triage.private_notes", value="brouillon", visibility="private", owner="triage")
store.write(agent="triage", key="incident.summary", value="Erreur checkout", visibility="shared")
store.create_handoff(
    from_agent="triage",
    to_agent="backend",
    keys=["triage.private_notes", "incident.summary"],
)

## À compléter

Ajoutez un patch atomique avec deux clés, puis déclenchez volontairement un conflit de version.

In [ ]:
# Votre code ici

# Corrections formateur

# Corrigé — Exercices — Sharing State

## Exercice 1 — Identifier les couches d'état

1. Brouillon interne d'un agent reviewer : **local state**.
2. Préférence utilisateur "réponds en français" : **long-term memory** si durable, sinon conversation state si temporaire.
3. Résumé validé d'un incident : **shared state**.
4. Historique des trois derniers messages : **conversation state**.
5. Plan d'exécution partagé entre agents : **shared state**.
6. Clé API utilisée par un outil : **secret/configuration**.
7. Statut `needs_clarification` d'un workflow : **conversation state** ou **shared state** selon le périmètre. Dans un workflow multi-agent, c'est généralement du shared state.

## Exercice 2 — Concevoir une entrée d'état

```json
{
  "incident.root_cause": {
    "value": "Timeout base de données sur la requête checkout",
    "version": 1,
    "owner": "backend",
    "visibility": "shared",
    "updated_by": "backend"
  }
}
```

## Exercice 3 — Détecter un conflit

L'écriture doit être refusée car l'agent pense modifier la version `1`, alors que la clé est déjà en version `2`.

Cela signifie qu'un autre agent a modifié la donnée entre la lecture et l'écriture.

Accepter cette écriture écraserait silencieusement une information plus récente.

## Exercice 4 — Snapshot filtré

L'agent `backend` peut voir :

```json
{
  "incident.summary": {
    "value": "erreur 500 sur /checkout",
    "visibility": "shared"
  },
  "incident.status": {
    "value": "investigating",
    "visibility": "public"
  }
}
```

Il ne peut pas voir :

```text
triage.private_notes
```

car la visibilité est `private` et le propriétaire est `triage`.

## Exercice 5 — Handoff minimal

```json
{
  "from": "triage",
  "to": "backend",
  "context": {
    "incident.summary": {
      "value": "erreur 500 sur /checkout",
      "version": 1
    },
    "incident.status": {
      "value": "investigating",
      "version": 1
    },
    "plan.next_action": {
      "value": "inspecter les logs API checkout",
      "version": 1
    }
  }
}
```

## Exercice 6 — Patch atomique

Un patch multi-clés doit être atomique pour éviter un état partiellement modifié.

Si une clé est mise à jour mais qu'une autre échoue, les agents suivants peuvent observer un état incohérent.

La règle correcte est :

```text
valider toutes les écritures, puis appliquer toutes les écritures
```

## Exercice 7 — Trace d'état

```json
{
  "event_id": 1,
  "agent": "coordinator",
  "operation": "write",
  "key": "incident.owner",
  "version": 1,
  "status": "applied"
}
```

## Exercice 8 — Lab

Le fichier `test_shared_state_store.py` montre une implémentation complète des vérifications demandées :

- création d'une clé privée ;
- restriction de lecture ;
- création d'une clé partagée ;
- snapshot filtré ;
- conflit de version ;
- journal d'événements.

# Corrigé — Questions d'entretien — Sharing State

## Question 1

Le `conversation state` représente l'état courant d'une interaction : intention, slots, dernier contexte utile.

Le `shared state` représente l'état partagé d'un workflow multi-agent : faits, décisions, résultats intermédiaires, plan.

La `long-term memory` représente des informations durables qui peuvent survivre à une conversation ou à un workflow : préférences utilisateur, souvenirs synthétiques, profil.

## Question 2

Une mémoire globale partagée est dangereuse car elle mélange :

- données temporaires ;
- données durables ;
- brouillons privés ;
- décisions validées ;
- secrets ;
- historique brut.

Elle augmente les risques de fuite, de conflit et d'incohérence.

## Question 3

Le versioning sert à détecter les écritures obsolètes.

Un agent ne doit pas écraser une donnée qu'il a lue dans une version ancienne.

## Question 4

Le compare-and-set consiste à écrire une valeur uniquement si la version actuelle correspond à la version attendue.

Exemple :

```text
write key=plan.next_step expected_version=2
```

Si la version actuelle n'est pas `2`, l'écriture est refusée.

## Question 5

Un handoff doit contenir :

- l'objectif ;
- les faits validés ;
- les contraintes ;
- les clés nécessaires ;
- les incertitudes utiles ;
- le statut actuel.

## Question 6

Un handoff ne doit pas contenir :

- toutes les traces ;
- l'intégralité de l'historique ;
- les brouillons privés ;
- les secrets ;
- les données non nécessaires ;
- les raisonnements internes.

## Question 7

On évite la lecture d'une donnée privée avec une vérification systématique des permissions à chaque lecture ou snapshot.

La règle minimale :

```text
private => owner only
shared/public => visible selon politique du workflow
```

## Question 8

Journaliser les changements d'état permet :

- d'auditer le workflow ;
- de comprendre les bugs ;
- de reconstruire la chronologie ;
- d'analyser les conflits ;
- d'alimenter l'observabilité.

## Question 9

On expose une partie du shared state comme ressource MCP quand un agent externe ou un client MCP doit lire un contexte stable et autorisé.

Il faut éviter d'exposer les données privées, les secrets et les brouillons.

## Question 10

Je testerais :

- les lectures autorisées ;
- les lectures refusées ;
- les écritures versionnées ;
- les conflits ;
- les patches atomiques ;
- le handoff minimal ;
- le journal d'événements ;
- la sérialisation JSON.

## Question 11

Deux agents qui modifient le même plan en parallèle peuvent :

- écraser une modification récente ;
- produire deux versions contradictoires ;
- générer un handoff incohérent ;
- masquer un conflit métier.

Le versioning et le compare-and-set permettent de détecter ce cas.

## Question 12

Pour passer en production, je remplacerais le store en mémoire par :

- PostgreSQL ou Redis selon les besoins ;
- transactions ;
- verrous optimistes ;
- logs persistants ;
- chiffrement des champs sensibles ;
- expiration des données temporaires ;
- métriques et traces ;
- contrôle d'accès plus fin.

# Corrigé — Challenge — Shared State Coordinator

## Solution attendue

Une solution correcte introduit un coordinateur qui possède le store partagé et n'autorise pas les agents à modifier directement un dictionnaire global.

## Architecture recommandée

```mermaid
flowchart LR
    U[Utilisateur] --> C[SharedStateCoordinator]
    C --> T[triage]
    C --> B[backend]
    C --> S[security]
    C --> R[reviewer]
    T -->|patch| Store[(SharedStateStore)]
    B -->|patch| Store
    S -->|patch| Store
    Store -->|handoff minimal| R
    Store --> Events[Event Log]
```

## Pseudo-code

```python
class SharedStateCoordinator:
    def __init__(self):
        self.store = SharedStateStore()

    def run(self, objective: str) -> dict:
        self.store.write(
            agent="coordinator",
            key="objective",
            value=objective,
            visibility="public"
        )

        self.store.apply_patch(
            agent="triage",
            writes={
                "incident.summary": {
                    "value": "Incident checkout à investiguer",
                    "visibility": "shared"
                }
            }
        )

        self.store.apply_patch(
            agent="backend",
            writes={
                "incident.root_cause": {
                    "value": "Timeout base de données probable",
                    "visibility": "shared"
                }
            }
        )

        handoff = self.store.create_handoff(
            from_agent="backend",
            to_agent="reviewer",
            keys=[
                "objective",
                "incident.summary",
                "incident.root_cause"
            ]
        )

        return {
            "status": "completed",
            "summary": "Incident analysé et prêt pour revue.",
            "handoff": handoff,
            "state": self.store.snapshot("reviewer"),
            "events": self.store.events()
        }
```

## Points évalués

Une bonne solution doit :

- centraliser le shared state ;
- empêcher les accès privés non autorisés ;
- utiliser des versions ;
- préserver l'atomicité des patches ;
- tracer les changements ;
- produire un handoff minimal ;
- garder la mémoire long terme hors du workflow temporaire.

## Bonus MCP Resource

Une méthode `export_as_mcp_resource` peut produire :

```json
{
  "uri": "state://incident/current",
  "mimeType": "application/json",
  "text": "{...}"
}
```

Elle doit filtrer les clés privées avant exposition.

# Review formateur — Sharing State

## Objectif de la revue

Vérifier que les apprenants comprennent que le shared state est un contrat d'architecture et non un simple dictionnaire global.

## Points à vérifier

- L'apprenant distingue local state, conversation state, shared state et memory.
- Il comprend pourquoi une écriture doit être versionnée.
- Il sait expliquer un conflit compare-and-set.
- Il comprend l'intérêt d'un patch atomique.
- Il ne transmet pas tout l'état lors d'un handoff.
- Il filtre les données privées.
- Il sait exploiter un event log.
- Il sait relier le shared state à MCP sans les confondre.

## Questions de relance

- Que se passe-t-il si deux agents écrivent sur `plan.next_step` ?
- Pourquoi le reviewer n'a-t-il pas besoin des notes privées du triage ?
- Que mettriez-vous dans Redis ? Que mettriez-vous dans PostgreSQL ?
- Quelles clés exposeriez-vous via MCP Resources ?
- Que supprimeriez-vous à la fin d'un workflow ?

## Erreurs fréquentes

1. Utiliser une variable globale partagée.
2. Confondre shared state et long-term memory.
3. Oublier les versions.
4. Oublier les permissions.
5. Mettre tout le journal d'événements dans le prompt.
6. Exposer des brouillons comme faits validés.
7. Rendre le handoff trop volumineux.

## Critères de validation

Un apprenant maîtrise la journée s'il peut :

- expliquer le modèle ;
- implémenter le store ;
- passer les tests ;
- justifier les règles de visibilité ;
- diagnostiquer un conflit ;
- produire un handoff minimal ;
- proposer une évolution production réaliste.

## Proposition d'amélioration

Ajouter ultérieurement un exercice optionnel avec Redis ou SQLite pour montrer la persistance transactionnelle, sans modifier la structure officielle de la journée.

In [ ]:
store = SharedStateStore()
store.write(agent="triage", key="plan.next_step", value="inspect logs")
store.write(agent="backend", key="plan.next_step", value="inspect database", expected_version=1)

try:
    store.write(agent="security", key="plan.next_step", value="rotate credentials", expected_version=1)
except ConflictError as error:
    print("Conflit détecté:", error)

store.summary()